In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [2]:
linked = gpd.read_parquet(
    "../data/processed/outage_crime_linked.parquet"
)

print("Rows:", len(linked))
print("Columns:", len(linked.columns))
print("CRS:", linked.crs)

Rows: 108030
Columns: 50
CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "de

In [3]:
print(linked.columns.tolist())

['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'descriptor_2', 'location_type', 'incident_zip', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'intersection_street_1', 'intersection_street_2', 'address_type', 'city', 'landmark', 'facility_type', 'status', 'due_date', 'resolution_description', 'resolution_action_updated_date', 'community_board', 'council_district', 'police_precinct', 'bbl', 'borough', 'x_coordinate_state_plane', 'y_coordinate_state_plane', 'open_data_channel_type', 'park_facility_name', 'park_borough', 'vehicle_type', 'taxi_company_borough', 'taxi_pick_up_location', 'bridge_highway_name', 'bridge_highway_direction', 'road_ramp', 'bridge_highway_segment', 'latitude', 'longitude', 'location', 'outage_duration_hours', 'geometry', 'h3_res9', 'h3_res10', 'crimes_during_outage_100m', 'crimes_during_outage_250m']


In [4]:
linked.head(3)

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,...,bridge_highway_segment,latitude,longitude,location,outage_duration_hours,geometry,h3_res9,h3_res10,crimes_during_outage_100m,crimes_during_outage_250m
0,70443136,2026-09-16 21:39:00,2026-09-17 09:34:00,DOT,Department of Transportation,Street Light Condition,Street Light Out,Location Type: Intersection,NaN,11201.0,...,NaN,40.695643,-73.997382,POINT (-73.997381886938 40.695643134051),11.916667,POINT (-73.99738 40.69564),892a1072d6fffff,8a2a1072d6dffff,0,0
1,70427817,2026-09-15 20:07:00,2026-09-17 09:17:00,DOT,Department of Transportation,Street Light Condition,Street Light Out,Location Type: Intersection,NaN,11221.0,...,NaN,40.693572,-73.922746,POINT (-73.92274641677 40.6935723826),37.166667,POINT (-73.92275 40.69357),892a100dd5bffff,8a2a100dd5affff,0,0
2,70423298,2026-09-15 15:54:00,2026-09-17 09:18:00,DOT,Department of Transportation,Street Light Condition,Street Light Out,Location Type: Intersection,NaN,11207.0,...,NaN,40.692119,-73.906600,POINT (-73.906600309246 40.692119407166),41.400000,POINT (-73.9066 40.69212),892a100dd13ffff,8a2a100dd127fff,0,0


In [5]:
linked[
    [
        "created_date",
        "closed_date",
        "crimes_during_outage_100m",
        "crimes_during_outage_250m",
        "geometry"
    ]
].head()

,created_date,closed_date,crimes_during_outage_100m,crimes_during_outage_250m,geometry
0,2026-09-16 21:39:00,2026-09-17 09:34:00,0,0,POINT (-73.99738 40.69564)
1,2026-09-15 20:07:00,2026-09-17 09:17:00,0,0,POINT (-73.92275 40.69357)
2,2026-09-15 15:54:00,2026-09-17 09:18:00,0,0,POINT (-73.9066 40.69212)
3,2026-09-15 12:47:00,2026-09-18 09:15:00,0,0,POINT (-73.98743 40.68714)
4,2026-09-15 09:35:00,2026-09-17 09:32:00,0,0,POINT (-73.9977 40.69498)


In [6]:
[h for h in linked.columns if "h3" in h.lower()]

['h3_res9', 'h3_res10']

In [7]:
print(
    [
        col for col in linked.columns
        if any(
            word in col.lower()
            for word in [
                "id",
                "unique",
                "object",
                "street"
            ]
        )
    ]
)

['unique_key', 'incident_zip', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'intersection_street_1', 'intersection_street_2', 'bridge_highway_name', 'bridge_highway_direction', 'bridge_highway_segment']


In [8]:
print(
    "Total outage records:",
    len(linked)
)

print(
    "Unique geometries:",
    linked.geometry.nunique()
)

Total outage records: 108030
Unique geometries: 38643


In [9]:
print(
    linked[
        [
            "created_date",
            "closed_date"
        ]
    ].describe()
)

                     created_date                 closed_date
count                      108030                      108030
mean   2023-11-18 21:16:35.901693  2023-12-10 22:04:50.861427
min           2020-01-01 00:45:00         2020-01-02 09:03:00
25%           2022-04-09 04:14:30         2022-05-09 13:59:15
50%           2024-03-06 04:21:00         2024-03-29 16:12:30
75%           2025-08-05 14:26:15         2025-08-23 16:38:00
max           2026-09-16 21:39:00         2026-09-18 09:15:00


In [10]:
# Check the important fields needed for control-area selection

key_columns = [
    "unique_key",
    "created_date",
    "closed_date",
    "h3_res9",
    "h3_res10",
    "crimes_during_outage_100m",
    "crimes_during_outage_250m",
    "geometry",
]

linked[key_columns].head()

,unique_key,created_date,closed_date,h3_res9,h3_res10,crimes_during_outage_100m,crimes_during_outage_250m,geometry
0,70443136,2026-09-16 21:39:00,2026-09-17 09:34:00,892a1072d6fffff,8a2a1072d6dffff,0,0,POINT (-73.99738 40.69564)
1,70427817,2026-09-15 20:07:00,2026-09-17 09:17:00,892a100dd5bffff,8a2a100dd5affff,0,0,POINT (-73.92275 40.69357)
2,70423298,2026-09-15 15:54:00,2026-09-17 09:18:00,892a100dd13ffff,8a2a100dd127fff,0,0,POINT (-73.9066 40.69212)
3,70424836,2026-09-15 12:47:00,2026-09-18 09:15:00,892a10729b3ffff,8a2a10729b37fff,0,0,POINT (-73.98743 40.68714)
4,70424835,2026-09-15 09:35:00,2026-09-17 09:32:00,892a10729d7ffff,8a2a10729d67fff,0,0,POINT (-73.9977 40.69498)


In [11]:
# Calculate outage duration in hours

linked["outage_duration_hours"] = (
    linked["closed_date"] - linked["created_date"]
).dt.total_seconds() / 3600

print(linked["outage_duration_hours"].describe())

count    108030.000000
mean        528.804155
std        1189.876400
min           0.500000
25%          45.333333
50%         160.383333
75%         286.733333
max        8759.866667
Name: outage_duration_hours, dtype: float64


In [12]:
# Check for invalid or unusual outage durations

print("Negative durations:",
      (linked["outage_duration_hours"] < 0).sum())

print("Zero-hour durations:",
      (linked["outage_duration_hours"] == 0).sum())

print("Outages longer than 7 days:",
      (linked["outage_duration_hours"] > 7 * 24).sum())

Negative durations: 0
Zero-hour durations: 0
Outages longer than 7 days: 51994


In [13]:
# Check the number of outages in each H3 resolution

h3_res9_counts = linked["h3_res9"].value_counts()
h3_res10_counts = linked["h3_res10"].value_counts()

print("Unique H3 resolution 9 cells:", h3_res9_counts.nunique())
print("Unique H3 resolution 10 cells:", h3_res10_counts.nunique())

print("\nH3 resolution 9:")
print(h3_res9_counts.describe())

print("\nH3 resolution 10:")
print(h3_res10_counts.describe())

Unique H3 resolution 9 cells: 88
Unique H3 resolution 10 cells: 53

H3 resolution 9:
count    6105.000000
mean       17.695332
std        14.298959
min         1.000000
25%         8.000000
50%        15.000000
75%        24.000000
max       396.000000
Name: count, dtype: float64

H3 resolution 10:
count    23701.000000
mean         4.558036
std          5.352240
min          1.000000
25%          2.000000
50%          3.000000
75%          6.000000
max        396.000000
Name: count, dtype: float64


In [14]:
# Show the most frequently occurring H3 cells

print("Top H3 res9 cells:")
print(h3_res9_counts.head(10))

print("\nTop H3 res10 cells:")
print(h3_res10_counts.head(10))

Top H3 res9 cells:
h3_res9
892a1008957ffff    396
892a100db63ffff    210
892a10728dbffff    157
892a100c1a7ffff    100
892a100aa27ffff     98
892a1008cb3ffff     89
892a100da5bffff     89
892a1001893ffff     89
892a1001c2fffff     88
892a107298bffff     87
Name: count, dtype: int64

Top H3 res10 cells:
h3_res10
8a2a10089547fff    396
8a2a100db607fff    207
8a2a10728d87fff    157
8a2a100c1a6ffff     81
8a2a100e372ffff     72
8a2a100cb0affff     60
8a2a100aa35ffff     57
8a2a100527affff     57
8a2a100f2b9ffff     55
8a2a1039c187fff     55
Name: count, dtype: int64


In [15]:
# Check for missing H3 identifiers

print("Missing h3_res9:", linked["h3_res9"].isna().sum())
print("Missing h3_res10:", linked["h3_res10"].isna().sum())

print("\nEmpty h3_res9:", (linked["h3_res9"] == "").sum())
print("Empty h3_res10:", (linked["h3_res10"] == "").sum())

Missing h3_res9: 0
Missing h3_res10: 0

Empty h3_res9: 0
Empty h3_res10: 0


In [16]:
# Check whether each H3 cell has a consistent spatial geometry

print(
    linked.groupby("h3_res9")["geometry"]
    .nunique()
    .describe()
)

print(
    linked.groupby("h3_res10")["geometry"]
    .nunique()
    .describe()
)

count    6105.000000
mean        6.329730
std         3.778415
min         1.000000
25%         4.000000
50%         6.000000
75%         9.000000
max        31.000000
Name: geometry, dtype: float64
count    23701.000000
mean         1.630438
std          0.920512
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         12.000000
Name: geometry, dtype: float64


In [17]:
# Create an outage-level analysis table

analysis_df = linked[
    [
        "unique_key",
        "created_date",
        "closed_date",
        "outage_duration_hours",
        "h3_res9",
        "h3_res10",
        "crimes_during_outage_100m",
        "crimes_during_outage_250m",
        "geometry",
    ]
].copy()

print("Rows:", len(analysis_df))
print("Columns:", len(analysis_df.columns))

analysis_df.head()

Rows: 108030
Columns: 9


,unique_key,created_date,closed_date,outage_duration_hours,h3_res9,h3_res10,crimes_during_outage_100m,crimes_during_outage_250m,geometry
0,70443136,2026-09-16 21:39:00,2026-09-17 09:34:00,11.916667,892a1072d6fffff,8a2a1072d6dffff,0,0,POINT (-73.99738 40.69564)
1,70427817,2026-09-15 20:07:00,2026-09-17 09:17:00,37.166667,892a100dd5bffff,8a2a100dd5affff,0,0,POINT (-73.92275 40.69357)
2,70423298,2026-09-15 15:54:00,2026-09-17 09:18:00,41.400000,892a100dd13ffff,8a2a100dd127fff,0,0,POINT (-73.9066 40.69212)
3,70424836,2026-09-15 12:47:00,2026-09-18 09:15:00,68.466667,892a10729b3ffff,8a2a10729b37fff,0,0,POINT (-73.98743 40.68714)
4,70424835,2026-09-15 09:35:00,2026-09-17 09:32:00,47.950000,892a10729d7ffff,8a2a10729d67fff,0,0,POINT (-73.9977 40.69498)


In [18]:
# Check uniqueness of outage IDs

print("Total rows:", len(analysis_df))
print("Unique outage IDs:", analysis_df["unique_key"].nunique())

print(
    "Duplicate outage IDs:",
    analysis_df["unique_key"].duplicated().sum()
)

Total rows: 108030
Unique outage IDs: 108030
Duplicate outage IDs: 0


In [19]:
analysis_df["created_date"] = pd.to_datetime(
    analysis_df["created_date"]
)

analysis_df["closed_date"] = pd.to_datetime(
    analysis_df["closed_date"]
)

analysis_df["outage_duration_hours"] = (
    analysis_df["closed_date"] - analysis_df["created_date"]
).dt.total_seconds() / 3600

In [20]:
# Convert to a projected CRS for distance-based operations

analysis_projected = analysis_df.to_crs("EPSG:2263")

# Create the 100 m treatment area

analysis_projected["treatment_geometry"] = (
    analysis_projected.geometry.buffer(100)
)

print("CRS:", analysis_projected.crs)
print("Rows:", len(analysis_projected))

CRS: EPSG:2263
Rows: 108030


In [21]:
analysis_projected["candidate_geometry"] = (
    analysis_projected.geometry.buffer(250)
)

print(
    analysis_projected[
        ["unique_key", "treatment_geometry", "candidate_geometry"]
    ].head()
)

   unique_key                                 treatment_geometry  \
0    70443136  POLYGON ((985076 192718.014, 985075.518 192708...   
1    70427817  POLYGON ((1005773 191973.014, 1005772.518 1919...   
2    70423298  POLYGON ((1010251 191448.014, 1010250.518 1914...   
3    70424836  POLYGON ((987836 189620.014, 987835.518 189610...   
4    70424835  POLYGON ((984988 192478.014, 984987.518 192468...   

                                  candidate_geometry  
0  POLYGON ((985226 192718.014, 985224.796 192693...  
1  POLYGON ((1005923 191973.014, 1005921.796 1919...  
2  POLYGON ((1010401 191448.014, 1010399.796 1914...  
3  POLYGON ((987986 189620.014, 987984.796 189595...  
4  POLYGON ((985138 192478.014, 985136.796 192453...  


In [22]:
control_candidates = analysis_projected[
    [
        "unique_key",
        "created_date",
        "closed_date",
        "h3_res9",
        "h3_res10",
        "geometry",
        "candidate_geometry",
    ]
].copy()

control_candidates = gpd.GeoDataFrame(
    control_candidates,
    geometry="candidate_geometry",
    crs="EPSG:2263"
)

print("Control candidate rows:", len(control_candidates))

Control candidate rows: 108030


In [23]:
outage_points = analysis_projected[
    [
        "unique_key",
        "created_date",
        "closed_date",
        "h3_res9",
        "h3_res10",
        "geometry",
    ]
].copy()

outage_points = gpd.GeoDataFrame(
    outage_points,
    geometry="geometry",
    crs="EPSG:2263"
)

In [24]:
nearby_controls = gpd.sjoin(
    outage_points,
    control_candidates[
        [
            "unique_key",
            "created_date",
            "closed_date",
            "h3_res9",
            "h3_res10",
            "candidate_geometry",
        ]
    ],
    how="inner",
    predicate="within"
)

print("Nearby candidate pairs:", len(nearby_controls))

Nearby candidate pairs: 1095956


In [25]:
nearby_controls = nearby_controls[
    nearby_controls["unique_key_left"]
    != nearby_controls["unique_key_right"]
].copy()

print(
    "Candidate pairs after removing self-matches:",
    len(nearby_controls)
)

Candidate pairs after removing self-matches: 987926


In [26]:
# Recover the actual geometry of the candidate control point
# using the index_right generated by the spatial join.

nearby_controls["control_geometry"] = (
    nearby_controls["index_right"]
    .map(outage_points["geometry"])
)

print(
    "Missing control geometries:",
    nearby_controls["control_geometry"].isna().sum()
)

Missing control geometries: 0


In [27]:
nearby_controls["distance_m"] = (
    nearby_controls.geometry
    .distance(nearby_controls["control_geometry"])
)

print(nearby_controls["distance_m"].describe())

count    987926.000000
mean         39.059190
std          77.021787
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         250.000000
Name: distance_m, dtype: float64


In [28]:
candidate_pairs = nearby_controls[
    (nearby_controls["distance_m"] > 100)
    &
    (nearby_controls["distance_m"] <= 250)
].copy()

print(
    "Candidates between 100 and 250 m:",
    len(candidate_pairs)
)

Candidates between 100 and 250 m: 186674


In [29]:
print(
    "Minimum distance:",
    candidate_pairs["distance_m"].min()
)

print(
    "Maximum distance:",
    candidate_pairs["distance_m"].max()
)

Minimum distance: 100.00499987701198
Maximum distance: 249.99999985052273


In [30]:
print(candidate_pairs.columns.tolist())

['unique_key_left', 'created_date_left', 'closed_date_left', 'h3_res9_left', 'h3_res10_left', 'geometry', 'index_right', 'unique_key_right', 'created_date_right', 'closed_date_right', 'h3_res9_right', 'h3_res10_right', 'control_geometry', 'distance_m']


In [31]:
# Check whether the treatment and candidate control outages
# overlap in time.

candidate_pairs["temporal_overlap"] = (
    (candidate_pairs["created_date_left"]
     <= candidate_pairs["closed_date_right"])
    &
    (candidate_pairs["closed_date_left"]
     >= candidate_pairs["created_date_right"])
)

print(
    candidate_pairs["temporal_overlap"].value_counts()
)

temporal_overlap
False    180272
True       6402
Name: count, dtype: int64


In [32]:
control_pairs = candidate_pairs[
    ~candidate_pairs["temporal_overlap"]
].copy()

print(
    "Non-overlapping control candidates:",
    len(control_pairs)
)

Non-overlapping control candidates: 180272


In [33]:
control_counts = (
    control_pairs
    .groupby("unique_key_left")
    .size()
)

print(control_counts.describe())

count    46124.000000
mean         3.908421
std          4.307247
min          1.000000
25%          1.000000
50%          2.000000
75%          5.000000
max         66.000000
dtype: float64


In [34]:
print(
    "Treatment outages with at least one eligible control:",
    control_counts.shape[0]
)

print(
    "Treatment outages without an eligible control:",
    len(analysis_df) - control_counts.shape[0]
)

Treatment outages with at least one eligible control: 46124
Treatment outages without an eligible control: 61906


In [35]:
control_pairs = control_pairs.sort_values(
    ["unique_key_left", "distance_m"]
)

In [36]:
selected_controls = (
    control_pairs
    .drop_duplicates(
        subset=["unique_key_left"],
        keep="first"
    )
    .copy()
)

print(
    "Selected treatment-control pairs:",
    len(selected_controls)
)

Selected treatment-control pairs: 46124


In [37]:
selected_controls[
    [
        "unique_key_left",
        "unique_key_right",
        "distance_m",
        "created_date_left",
        "closed_date_left",
        "created_date_right",
        "closed_date_right",
    ]
].head(10)

,unique_key_left,unique_key_right,distance_m,created_date_left,closed_date_left,created_date_right,closed_date_right
108026,45284934,48296388,176.139150,2020-01-01 11:04:00,2020-01-04 00:05:00,2020-12-01 16:20:00,2020-12-03 01:23:00
108028,45284935,64135128,132.457540,2020-01-01 06:55:00,2020-01-03 10:54:00,2025-02-20 15:25:00,2025-02-26 03:25:00
108019,45285212,64845276,135.203550,2020-01-01 18:19:00,2020-01-06 09:16:00,2025-05-04 21:41:00,2025-05-13 15:22:00
108009,45285700,48430637,106.470653,2020-01-01 20:11:00,2020-01-06 09:27:00,2020-12-16 09:54:00,2020-12-18 08:59:00
108023,45286571,53726952,149.083869,2020-01-01 15:26:00,2020-01-03 12:15:00,2022-03-23 17:18:00,2022-03-24 11:27:00
108014,45286578,68429764,143.377125,2020-01-01 19:30:00,2020-01-02 23:25:00,2026-03-23 20:20:00,2026-03-26 08:05:00
108025,45287019,47293303,233.931614,2020-01-01 11:40:00,2020-01-06 09:15:00,2020-08-19 22:03:00,2020-08-20 08:19:00
108011,45287028,48318022,149.753130,2020-01-01 20:00:00,2020-01-02 21:00:00,2020-12-03 21:00:00,2020-12-11 08:33:00
108016,45287316,67462679,164.268074,2020-01-01 18:58:00,2020-01-03 13:52:00,2026-01-11 18:34:00,2026-01-29 08:33:00
108027,45287354,49563721,191.316492,2020-01-01 11:03:00,2020-01-02 11:57:00,2021-01-19 01:52:00,2021-01-19 14:34:00


In [38]:
selected_controls = selected_controls.rename(
    columns={
        "unique_key_left": "treatment_key",
        "unique_key_right": "control_key"
    }
)

In [39]:
selected_controls[
    [
        "treatment_key",
        "control_key",
        "distance_m"
    ]
].head()

,treatment_key,control_key,distance_m
108026,45284934,48296388,176.139150
108028,45284935,64135128,132.457540
108019,45285212,64845276,135.203550
108009,45285700,48430637,106.470653
108023,45286571,53726952,149.083869


In [40]:
treatment_data = analysis_df[
    [
        "unique_key",
        "created_date",
        "closed_date",
        "outage_duration_hours",
        "h3_res9",
        "h3_res10",
        "crimes_during_outage_100m",
        "crimes_during_outage_250m",
    ]
].copy()

treatment_data = treatment_data.rename(
    columns={
        "unique_key": "treatment_key",
        "crimes_during_outage_100m":
            "treatment_crime_100m",
        "crimes_during_outage_250m":
            "treatment_crime_250m",
    }
)

In [41]:
control_data = analysis_df[
    [
        "unique_key",
        "crimes_during_outage_100m",
        "crimes_during_outage_250m",
    ]
].copy()

control_data = control_data.rename(
    columns={
        "unique_key": "control_key",
        "crimes_during_outage_100m":
            "control_crime_100m",
        "crimes_during_outage_250m":
            "control_crime_250m",
    }
)

In [42]:
paired_analysis = selected_controls.merge(
    treatment_data,
    on="treatment_key",
    how="inner"
)

In [43]:
paired_analysis = paired_analysis.merge(
    control_data,
    on="control_key",
    how="inner"
)

In [44]:
print("Paired analysis rows:", len(paired_analysis))

print(
    paired_analysis[
        [
            "treatment_key",
            "control_key",
            "distance_m",
            "treatment_crime_100m",
            "control_crime_100m",
            "treatment_crime_250m",
            "control_crime_250m",
        ]
    ].head(10)
)

Paired analysis rows: 46124
   treatment_key  control_key  distance_m  treatment_crime_100m  \
0       45284934     48296388  176.139150                     0   
1       45284935     64135128  132.457540                     0   
2       45285212     64845276  135.203550                     0   
3       45285700     48430637  106.470653                     0   
4       45286571     53726952  149.083869                     0   
5       45286578     68429764  143.377125                     0   
6       45287019     47293303  233.931614                     0   
7       45287028     48318022  149.753130                     0   
8       45287316     67462679  164.268074                     0   
9       45287354     49563721  191.316492                     0   

   control_crime_100m  treatment_crime_250m  control_crime_250m  
0                   0                     0                   0  
1                   0                     0                   0  
2                   0               

In [45]:
paired_analysis["crime_difference_100m"] = (
    paired_analysis["treatment_crime_100m"]
    - paired_analysis["control_crime_100m"]
)

paired_analysis["crime_difference_250m"] = (
    paired_analysis["treatment_crime_250m"]
    - paired_analysis["control_crime_250m"]
)

In [46]:
paired_analysis[
    [
        "treatment_key",
        "control_key",
        "distance_m",
        "treatment_crime_100m",
        "control_crime_100m",
        "crime_difference_100m",
        "treatment_crime_250m",
        "control_crime_250m",
        "crime_difference_250m",
    ]
].head(10)

,treatment_key,control_key,distance_m,treatment_crime_100m,control_crime_100m,crime_difference_100m,treatment_crime_250m,control_crime_250m,crime_difference_250m
0,45284934,48296388,176.139150,0,0,0,0,0,0
1,45284935,64135128,132.457540,0,0,0,0,0,0
2,45285212,64845276,135.203550,0,0,0,0,0,0
3,45285700,48430637,106.470653,0,0,0,0,0,0
4,45286571,53726952,149.083869,0,0,0,0,0,0
5,45286578,68429764,143.377125,0,0,0,0,0,0
6,45287019,47293303,233.931614,0,0,0,0,0,0
7,45287028,48318022,149.753130,0,0,0,0,0,0
8,45287316,67462679,164.268074,0,0,0,0,0,0
9,45287354,49563721,191.316492,0,0,0,0,0,0


In [47]:
print("===== CONTROL PAIR VALIDATION =====")

print(
    "Number of pairs:",
    len(paired_analysis)
)

print(
    "Unique treatment outages:",
    paired_analysis["treatment_key"].nunique()
)

print(
    "Unique controls:",
    paired_analysis["control_key"].nunique()
)

print(
    "Self matches:",
    (
        paired_analysis["treatment_key"]
        == paired_analysis["control_key"]
    ).sum()
)

print(
    "Minimum distance:",
    paired_analysis["distance_m"].min()
)

print(
    "Maximum distance:",
    paired_analysis["distance_m"].max()
)

===== CONTROL PAIR VALIDATION =====
Number of pairs: 46124
Unique treatment outages: 46124
Unique controls: 14449
Self matches: 0
Minimum distance: 100.00499987701198
Maximum distance: 249.99999985052273


In [48]:
print(
    "Controls at or below 100m:",
    (paired_analysis["distance_m"] <= 100).sum()
)

print(
    "Controls above 250m:",
    (paired_analysis["distance_m"] > 250).sum()
)

Controls at or below 100m: 0
Controls above 250m: 0


In [49]:
print("Missing treatment crime 100m:",
      paired_analysis["treatment_crime_100m"].isna().sum())

print("Missing control crime 100m:",
      paired_analysis["control_crime_100m"].isna().sum())

print("Missing treatment crime 250m:",
      paired_analysis["treatment_crime_250m"].isna().sum())

print("Missing control crime 250m:",
      paired_analysis["control_crime_250m"].isna().sum())

Missing treatment crime 100m: 0
Missing control crime 100m: 0
Missing treatment crime 250m: 0
Missing control crime 250m: 0


In [50]:
print(
    paired_analysis["distance_m"].describe()
)

count    46124.000000
mean       178.294082
std         45.930669
min        100.005000
25%        136.970800
50%        179.401784
75%        219.107280
max        250.000000
Name: distance_m, dtype: float64


In [51]:
OUTPUT_FILE = Path(
    "../data/processed/control_area_pairs.parquet"
)

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

paired_analysis.to_parquet(
    OUTPUT_FILE,
    index=False
)

print(
    f"Saved Stage 7 output to {OUTPUT_FILE}"
)

Saved Stage 7 output to ..\data\processed\control_area_pairs.parquet


In [52]:
print("====================================")
print("       STAGE 7 FINAL SUMMARY")
print("====================================")

print(
    "Original outage records:",
    len(analysis_df)
)

print(
    "Treatment-control pairs:",
    len(paired_analysis)
)

print(
    "Unique treatment outages:",
    paired_analysis["treatment_key"].nunique()
)

print(
    "Unique control outages:",
    paired_analysis["control_key"].nunique()
)

print(
    "Self matches:",
    (
        paired_analysis["treatment_key"]
        == paired_analysis["control_key"]
    ).sum()
)

print(
    "Control distance (min):",
    paired_analysis["distance_m"].min()
)

print(
    "Control distance (max):",
    paired_analysis["distance_m"].max()
)

print(
    "Average control distance:",
    paired_analysis["distance_m"].mean()
)

print("====================================")

       STAGE 7 FINAL SUMMARY
Original outage records: 108030
Treatment-control pairs: 46124
Unique treatment outages: 46124
Unique control outages: 14449
Self matches: 0
Control distance (min): 100.00499987701198
Control distance (max): 249.99999985052273
Average control distance: 178.29408175910328


In [53]:
# Check how frequently each control outage is reused

control_reuse = (
    paired_analysis["control_key"]
    .value_counts()
)

print("Unique controls:", control_reuse.shape[0])

print("\nControl reuse distribution:")
print(control_reuse.describe())

print("\nControls used more than once:",
      (control_reuse > 1).sum())

print("\nControls used more than 5 times:",
      (control_reuse > 5).sum())

print("\nControls used more than 10 times:",
      (control_reuse > 10).sum())

print("\nMost frequently reused controls:")
print(control_reuse.head(20))

Unique controls: 14449

Control reuse distribution:
count    14449.000000
mean         3.192193
std          3.394503
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max         53.000000
Name: count, dtype: float64

Controls used more than once: 8650

Controls used more than 5 times: 2241

Controls used more than 10 times: 574

Most frequently reused controls:
control_key
70019443    53
65369700    52
65993087    48
59793318    42
70227557    37
45566244    37
64789186    36
63362522    35
49913258    33
51090243    33
52597631    32
46375450    32
57514181    32
67408562    32
55696868    31
51540113    31
68264239    30
64856331    30
69986599    30
57111675    30
Name: count, dtype: int64


In [54]:
# Check how many treatment outages share the same control

print(
    "Maximum number of times one control is reused:",
    control_reuse.max()
)

Maximum number of times one control is reused: 53
